# Подбор гиперпараметров LightGBM (Optuna)
Нужно: pip install optuna lightgbm
Всё считается на dev. Hold-out не трогаем.

**Протокол**
1. Поиск: 3 фолда, подвыборка dev, learning_rate=0.1, early stopping. Быстро, но грубо.
2. Подбираем число деревьев для финальной конфигурации (lr=0.05) на полном dev.
3. Честная оценка: те же 5 фолдов, что в 05, фиксированное число деревьев, без early stopping.
   Так мы сравниваем с базовым LightGBM по тем же правилам.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

sys.path.insert(0, "..")
from src.data import load_features, split_dev_holdout
from src.features import ID_COL, TARGET_COL
from src.metrics import report
from src.models import RANDOM_STATE, cv_predict, make_lgbm, to_lgbm
from src.tuning import BASELINE_PARAMS, find_n_estimators, make_objective

PROCESSED = Path("../data/processed")
MODELS = Path("../models")
MODELS.mkdir(exist_ok=True)

# Настройки скорости. Если слишком долго: уменьши TUNE_FRAC и N_TRIALS.
TUNE_FRAC = 0.5       # доля dev для поиска
SEARCH_LR = 0.1       # быстрый темп на этапе поиска
N_TRIALS = 40
TIMEOUT_SEC = 60 * 60  # страховка: не дольше часа за один запуск
FINAL_LR = 0.05       # темп для финальной конфигурации

In [ ]:
df = load_features(PROCESSED / "train_features.parquet")
dev, _ = split_dev_holdout(df, PROCESSED / "holdout_ids.csv")  # hold-out отбрасываем
y = dev[TARGET_COL]
X = to_lgbm(dev.drop(columns=[TARGET_COL, ID_COL]))
print("dev:", X.shape)

# подвыборка для поиска (стратифицированная)
X_tune, _, y_tune, _ = train_test_split(
    X, y, train_size=TUNE_FRAC, stratify=y, random_state=RANDOM_STATE
)
X_tune, y_tune = X_tune.reset_index(drop=True), y_tune.reset_index(drop=True)
print("для поиска:", X_tune.shape, f"| доля дефолтов {y_tune.mean():.4f}")

## 1. Поиск
Первым испытанием кладём параметры нашей базовой модели: так видно, обгоняет ли подбор
базу в одинаковом протоколе. Результаты пишутся в SQLite, поэтому после падения ядра
можно перезапустить ячейку, и поиск продолжится с места остановки.

In [ ]:
study = optuna.create_study(
    direction="maximize",
    study_name="lgbm_auc",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8),
    storage=f"sqlite:///{(PROCESSED / 'optuna_lgbm.db').as_posix()}",
    load_if_exists=True,
)
if len(study.trials) == 0:
    study.enqueue_trial(BASELINE_PARAMS)

objective = make_objective(X_tune, y_tune, lr=SEARCH_LR, n_splits=3)
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC)

In [ ]:
done = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"Завершено: {len(done)} | обрезано pruner'ом: {len(pruned)}")

baseline_auc = study.trials[0].value
print(f"Базовые параметры в этом протоколе: {baseline_auc:.4f}")
print(f"Лучший результат подбора:           {study.best_value:.4f}")
print("Лучшие параметры:")
print(json.dumps(study.best_params, indent=2))

In [ ]:
# История и важность параметров
vals = [t.value for t in done]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(len(vals)), vals, "o", alpha=0.5, label="испытание")
ax.plot(range(len(vals)), np.maximum.accumulate(vals), label="лучшее на данный момент")
ax.axhline(baseline_auc, color="k", ls="--", alpha=0.4, label="базовые параметры")
ax.set(xlabel="завершённое испытание", ylabel="AUC (3 фолда)", title="Ход подбора")
ax.legend()
plt.show()

print(pd.Series(optuna.importance.get_param_importances(study)).round(3))

In [ ]:
# Топ-10 конфигураций
top = (study.trials_dataframe()
       .query("state == 'COMPLETE'")
       .sort_values("value", ascending=False)
       .head(10))
top[["number", "value"] + [c for c in top.columns if c.startswith("params_")]].round(4)

## 2. Число деревьев для финальной конфигурации (lr = 0.05, полный dev)

In [ ]:
best = study.best_params
iters = find_n_estimators(X, y, best, lr=FINAL_LR, n_splits=5)
n_final = int(np.mean(iters))
print("Оптимальное число деревьев по фолдам:", iters, "-> берём", n_final)

## 3. Честное сравнение с базовой моделью
Те же 5 фолдов, что в 05; число деревьев фиксировано, early stopping не используется.

In [ ]:
oof_tuned, fold_tuned = cv_predict(
    lambda: make_lgbm(n_estimators=n_final, learning_rate=FINAL_LR, **best), X, y
)

# базовые фолды восстанавливаем из сохранённых OOF (фолды те же: тот же random_state и порядок dev)
oof_saved = pd.read_parquet(PROCESSED / "oof_dev.parquet")
assert (oof_saved[ID_COL].values == dev[ID_COL].values).all(), "порядок строк dev изменился"
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
fold_base = np.array([
    roc_auc_score(y.iloc[va], oof_saved["LightGBM"].iloc[va]) for _, va in skf.split(X, y)
])

summary = pd.DataFrame([
    {**report("LightGBM (базовый)", y, oof_saved["LightGBM"].values),
     "AUC folds std": round(fold_base.std(), 4)},
    {**report("LightGBM (tuned)", y, oof_tuned),
     "AUC folds std": round(fold_tuned.std(), 4)},
]).set_index("model")
summary

In [ ]:
d = fold_tuned - fold_base
print("Разница по фолдам:", d.round(4))
print(f"mean {d.mean():.4f} | min {d.min():.4f} | max {d.max():.4f}")

## 4. Сохраняем результат

In [ ]:
oof_saved["LightGBM_tuned"] = oof_tuned
oof_saved.to_parquet(PROCESSED / "oof_dev.parquet", index=False)

final = {"params": best, "n_estimators": n_final, "learning_rate": FINAL_LR,
         "cv_auc_mean": round(float(fold_tuned.mean()), 4)}
(MODELS / "lgbm_best_params.json").write_text(json.dumps(final, indent=2))
print(json.dumps(final, indent=2))

Читаем так: если `min` разницы по фолдам > 0, улучшение стабильно. Если подбор дал меньше
+0.002 или нестабильно, значит, стартовые параметры уже были близки к оптимуму. Это
нормальный результат, о нём так и напишем в README.